# Pipeline Reference

Genel akis, GroundTruthcalendara hisse portfolyonun icerisindeki hisse senedi close fiyat bilgilerinin tamamini, portfolyo agirligini vermek, sonra tahmin yapilacak zaman dilimini vermek, feature builders ile feature olusturmak, model icerisine vermek 


**Flow:** `GroundTruthCalendar` → `build_*_features` → `*_train` (model + `ResidualBank` + `EwmaVarianceBank`) → `multi_day_var_from_*` → `calendar.backtest`


In [ ]:
import numpy as np
import pandas as pd
import data_management as dm
from core import feature_builders as fb
import tree_models as tm
from graph_helpers import plot_var_vs_loss

## 1. Data and portfolio

| Parameter | Meaning | Note |
|---|---|---|
| `TICKERS` | Portfolio constituents | The csv must have a column of the same name, otherwise `ValueError` |
| `WEIGHTS` | Weights (dict or Series) | Automatically normalized to sum to 1; missing ticker / non-finite value / zero sum raises an error |
| `DATA_PATH` | **Raw price** csv (`Date` index) | The calendar applies `pct_change()` itself, so it must be given **prices, not returns** (otherwise returns of returns are computed) |

In [ ]:
# >>> TUNABLE
TICKERS = ["AAPL", "JPM", "XOM", "KO"]
WEIGHTS = {"AAPL": 0.25, "JPM": 0.25, "XOM": 0.25, "KO": 0.25}
DATA_PATH = "./data_general/4_ticker_data.csv"
# <<<

df = pd.read_csv(DATA_PATH, parse_dates=["Date"], index_col="Date")   # raw prices
df.tail(3)

## 2. Calendar (`GroundTruthCalendar`)


The class determines the frequency: `DailyGroundTruthCalendar` (24h), `HourlyGroundTruthCalendar` (1h), `MinuteGroundTruthCalendar` (1m). All share the same constructor signature.

| Parameter | Meaning | Rules |
|---|---|---|
| `train_start`, `train_end`, `forecast_start`, `forecast_end` | Training / forecast windows | **All four must be actual bars in the data** (`KeyError` otherwise). `train_start <= train_end < forecast_start <= forecast_end` (no look-ahead). Expected weekdays: starts on Monday, ends on Friday; otherwise only a `[WARNING]` is printed |
| `drop_market_closed` | Weekdays missing from the data (holidays) | `True`: print a warning and skip them; `False`: raise `ValueError` (daily frequency) |

Derived attributes: `calendar.train_data`, `calendar.forecast_data`, `calendar.forecast_dates`, `calendar.weights`, `calendar.tickers`, `calendar.frequency`.

The window lengths directly determine **how many residuals accumulate**, which sets the upper bound of `multi_day_var_calculation_window_size` (see Section 5).

In [ ]:
# >>> TUNABLE
TRAIN_START    = "2024-06-03"   # Monday
TRAIN_END      = "2024-11-29"   # Friday
FORECAST_START = "2024-12-02"   # Monday
FORECAST_END   = "2024-12-31"
DROP_MARKET_CLOSED = True
# <<<

calendar = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start=TRAIN_START, train_end=TRAIN_END,
    forecast_start=FORECAST_START, forecast_end=FORECAST_END,
    drop_market_closed=DROP_MARKET_CLOSED,
)

## 3. Feature builders

`build_long_features` and `build_wide_features` take **the same four parameters** and return `(TrainingFeature, ForecastFeature)`.

| Parameter | Default | Meaning |
|---|---|---|
| `windows` | `(5, 10, 21)` | For each `w`: `real_vol_w` (rolling std), `mean_abs_w`, `skew_w`. **As many initial bars as the largest window are lost to warmup** (the start of the training set shrinks) |
| `ewma_span` | `21` | EWMA span for `*_ewma_vol` and (long only) `market_ewma_vol`. Also contributes to warmup |
| `sq_return_lags` | `None` | E.g. `(1, 2, 3)` adds `r2_lag_k` (lagged squared return) features. Drops as many extra rows as the largest lag |
| `target_scale` | `1.0` | `y = (r_next)² * target_scale`. Squared returns are very small, so increase this to rescale the target. Stored as `TrainingFeature.target_scale`; LSTM predictions are mapped back with `/ target_scale` |

**Long vs wide**
- **long:** one row per (bar, ticker), all tickers pooled into one model; target = that ticker's next-bar squared return. Portfolio variance is assembled by `EwmaVarianceBank` (`wᵀ Σ w`).
- **wide:** one row per bar; features for every ticker plus `portfolio_return`; target = the portfolio's next-bar squared return.

> Features are built **against a specific calendar**. If the calendar changes, the features must be rebuilt; features built from one calendar must not be used with another.

In [ ]:
# >>> TUNABLE
WINDOWS        = (5, 10, 21)
EWMA_SPAN      = 21
SQ_RETURN_LAGS = None     # e.g. (1, 2, 3)
TARGET_SCALE   = 1.0
# <<<

feat_kwargs = dict(windows=WINDOWS, ewma_span=EWMA_SPAN, sq_return_lags=SQ_RETURN_LAGS, target_scale=TARGET_SCALE)

training_feature_long, forecast_feature_long = fb.build_long_features(calendar, **feat_kwargs)
training_feature_wide, forecast_feature_wide = fb.build_wide_features(calendar, **feat_kwargs)

### Contents of the feature objects (inspection only, no parameters)

| Field | Meaning |
|---|---|
| `X_in_sample`, `y_in_sample` | Training rows whose target is known |
| `X_out_sample_last_index` | Last row of the training window (target not yet known); used for the "pending" prediction of the first forecast bar |
| `X_in_sample_id_cols`, `X_out_sample_last_index_id_cols` | Long only: `Date`, `ticker` per row |
| `feature_dates` | Dates actually covered by the features (can start later than `train_start` because of warmup) |
| `DANGEROUS_r_next_*` | Realized next-bar return. **For residuals / backtesting only**; it must not be used as a model feature |

In [ ]:
training_feature_long.X_in_sample_id_cols.tail(3)

In [ ]:
training_feature_long.X_in_sample.tail(3)

In [ ]:
training_feature_long.X_out_sample_last_index_id_cols

In [ ]:
training_feature_wide.X_in_sample.tail(3)

## 4. Model training

Three entry points: `xgboost_train`, `random_forest_train` (both thin wrappers around `tree_train`), and `tree_train(calendar, training_feature, model_constructor, model_name, ...)` directly (to plug in any other sklearn-style regressor).

**Shared (positional) signature:** `(calendar, training_feature, rolling_mu_hat_window_size, residual_bank_setting_mu_mode="constant", residual_bank_setting_default_mu_hat=0.0, **model_kwargs)`

If `training_feature.type` is `"long"` the long path is used, otherwise the wide path, automatically.

> ⚠️ **Known limitations (current state of `tree_models.py`):**
> 1. **The wide path does not run:** in `_tree_train_wide`, `ResidualBank(...)` is not given the window at construction, and the call `bank.training_safe_bulk_insert(..., rolling_mu_hat_window_size=...)` passes a kwarg that `training_safe_bulk_insert` does not accept → `TypeError`.
> 2. **`mu_mode="rolling"` does not run on the long path:** `_tree_train_long` first builds a placeholder `ResidualBank` without a window → `ValueError: mu_mode='rolling' requires a positive rolling_mu_hat_window_size, got: None`.
>
> The combination that currently works is **long + `mu_mode="constant"`**.

| Parameter | Default | Meaning |
|---|---|---|
| `rolling_mu_hat_window_size` | **required** | Last-N-bars window used to compute `mu_hat` (drift) in `ResidualBank`. It affects results **only when `mu_mode="rolling"`**; with `"constant"` its value is not read (but a positive int is still required by the signature) |
| `residual_bank_setting_mu_mode` | `"constant"` | `"constant"`: `mu_hat = default_mu_hat` for every bar. `"rolling"`: `mu_hat_t` = mean return over the last N bars **before** t (t excluded, no look-ahead) |
| `residual_bank_setting_default_mu_hat` | `0.0` | The fixed drift under `"constant"`; the initial value of the seed under `"rolling"` |
| `**model_kwargs` | see below | Passed straight to `XGBRegressor` / `RandomForestRegressor`; **overrides** the defaults |

**Model defaults**

| Model | Defaults |
|---|---|
| xgboost | `n_estimators=100, max_depth=3, learning_rate=0.05, verbosity=0, n_jobs=1` |
| random forest | `n_estimators=200, max_depth=5, random_state=0` |

`standardized_residual = (actual_return - mu_hat) / sqrt(predicted_variance)`; VaR is derived from a quantile of these residuals.

In [ ]:
# >>> TUNABLE
ROLLING_MU_HAT_WINDOW_SIZE = 20         # only meaningful when mu_mode="rolling"
MU_MODE        = "constant"             # "constant" | "rolling"  (rolling: see the known limitations above)
DEFAULT_MU_HAT = 0.0

XGB_KWARGS = dict(n_estimators=100, max_depth=3, learning_rate=0.05)   # overrides the defaults
RF_KWARGS  = dict(n_estimators=200, max_depth=5, random_state=0)
# <<<

train_kwargs = dict(
    residual_bank_setting_mu_mode=MU_MODE,
    residual_bank_setting_default_mu_hat=DEFAULT_MU_HAT,
)

model_xgb_long = tm.xgboost_train(calendar, training_feature_long, ROLLING_MU_HAT_WINDOW_SIZE, **train_kwargs, **XGB_KWARGS)
model_rf_long  = tm.random_forest_train(calendar, training_feature_long, ROLLING_MU_HAT_WINDOW_SIZE, **train_kwargs, **RF_KWARGS)

In [ ]:
# training output: model.residual_bank_in_sample  (ResidualBank)
#                  model.ewma_variance_bank_in_sample (long only; EwmaVarianceBank)
model_xgb_long.residual_bank_in_sample.to_dataframe().tail()

In [ ]:
model_xgb_long.ewma_variance_bank_in_sample.to_dataframe().tail()

### 4b. Long-path parameters not exposed by `tree_train`

In the chain `tree_train` → `_tree_train_long` → `long_format_train_seed(...)`, the following two parameters are **not forwarded from `tree_train`** and stay at the `long_format_train_seed` defaults:

| Parameter | Default | Meaning |
|---|---|---|
| `ewma_lam` | `0.94` | RiskMetrics EWMA decay (for correlation / covariance). Larger means slower forgetting |
| `ewma_min_periods` | `30` | EWMA burn-in: the sample covariance of the first N training bars seeds the recursion. `train_data` needs at least this many bars. The `usable_train_dates` threshold is `ewma_min_periods - 1`; that many bars do not enter the bank |

To change them, call `long_format_train_seed` directly and assign the resulting banks to the model:

In [ ]:
# >>> TUNABLE (long path only)
EWMA_LAM         = 0.94
EWMA_MIN_PERIODS = 30
# <<<

import copy
model_xgb_long_custom_ewma = copy.copy(model_xgb_long)   # a shallow copy is enough: only the two bank attributes are reassigned
(model_xgb_long_custom_ewma.residual_bank_in_sample,
 model_xgb_long_custom_ewma.ewma_variance_bank_in_sample) = tm.long_format_train_seed(
    model_xgb_long_custom_ewma, calendar, training_feature_long, ROLLING_MU_HAT_WINDOW_SIZE,
    ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS,
    residual_bank_setting_mu_mode=MU_MODE, residual_bank_setting_default_mu_hat=DEFAULT_MU_HAT,
)

## 5. Forecast: producing VaR

Two functions with the same logic; which one applies depends on the feature type:

| Feature type | Function | Returns |
|---|---|---|
| long | `multi_day_var_from_long_format_model(model, calendar, forecast_feature, window, conf, residual_bank, ewma_bank, run_key=None)` | `(var_df, residual_bank, ewma_bank)` |
| wide | `multi_day_var_from_pretrained_model(model, forecast_feature, calendar, window, conf, run_key=None)` | `var_df` (**a single object**, and a different argument order) |

| Parameter | Meaning |
|---|---|
| `multi_day_var_calculation_window_size` | VaR quantile window: for day t, the `(1 - var_confidence)` quantile of the last N standardized residuals **before** t. **Must be compatible with the history in the `ResidualBank`**: the bank holds (training outcomes + forecast outcomes) residuals; if that is not enough, `ValueError("Not enough prior residual history ...")` is raised. Small N adapts quickly but is noisy; large N is stable but slow |
| `var_confidence` | E.g. `0.95` for 95% VaR, i.e. the lower 5% quantile of the residual distribution |
| `initial_residual_bank` / `initial_ewma_variance_bank` | Usually `model.residual_bank_in_sample` / `model.ewma_variance_bank_in_sample`. The function works on a **copy**, so the model's own banks are not modified |
| `run_key` | Key under which the result is stored in `calendar.var_prediction_runs`. Defaults to `model.model_name`, so **re-running the same model type with different settings overwrites the earlier run**; give each experiment a unique key |

`forecast_dates[0]` is never scored (its feature bar is the last training bar); `calendar.forecast_pairs()` gives the pairing. So `n` forecast bars yield `n-1` VaR values.

`rolling_mu_hat_window_size` is **not an argument** at this stage; it arrives embedded in the bank.

In [ ]:
# >>> TUNABLE
MULTI_DAY_VAR_WINDOW = 90     # quantile window (independent; need not equal ROLLING_MU_HAT_WINDOW_SIZE)
VAR_CONFIDENCE       = 0.95
# <<<

var_df, residual_bank, ewma_bank = dm.multi_day_var_from_long_format_model(
    model_xgb_long, calendar, forecast_feature_long,
    MULTI_DAY_VAR_WINDOW, VAR_CONFIDENCE,
    model_xgb_long.residual_bank_in_sample, model_xgb_long.ewma_variance_bank_in_sample,
    run_key="xgb_long",
)
var_df.head()

In [ ]:
plot_var_vs_loss(var_df, residual_bank)

### Comparison with random forest (same window, separate `run_key`)

In [ ]:
var_df_rf, residual_bank_rf, _ = dm.multi_day_var_from_long_format_model(
    model_rf_long, calendar, forecast_feature_long,
    MULTI_DAY_VAR_WINDOW, VAR_CONFIDENCE,
    model_rf_long.residual_bank_in_sample, model_rf_long.ewma_variance_bank_in_sample,
    run_key="rf_long",
)
plot_var_vs_loss(var_df_rf, residual_bank_rf, title="RF: VaR Forecast vs Realized Loss")

### Window sensitivity

The same trained model with different `multi_day_var_calculation_window_size` values. No retraining is needed (the window only matters at forecast time). Each experiment uses a different `run_key`.

In [ ]:
# >>> TUNABLE
WINDOW_GRID = [20, 60, 90]
# <<<

window_results = {}
for w in WINDOW_GRID:
    try:
        v, rb, _ = dm.multi_day_var_from_long_format_model(
            model_xgb_long, calendar, forecast_feature_long, w, VAR_CONFIDENCE,
            model_xgb_long.residual_bank_in_sample, model_xgb_long.ewma_variance_bank_in_sample,
            run_key=f"xgb_long_w{w}",
        )
        window_results[w] = v
    except ValueError as e:
        print(f"window={w}: {e}")

## 6. Wide flow

> ⚠️ `tree_models._tree_train_wide` currently fails (known limitation 1 in Section 4), so the first cell below raises `TypeError`. The cells are written to run unchanged once that limitation is resolved.

Same parameters as above; the differences: `training_feature_wide` / `forecast_feature_wide` are used, there is no EWMA bank (`ewma_variance_bank_in_sample is None`), and the forecast function is `multi_day_var_from_pretrained_model`.

In [ ]:
model_xgb_wide = tm.xgboost_train(calendar, training_feature_wide, ROLLING_MU_HAT_WINDOW_SIZE, **train_kwargs, **XGB_KWARGS)

var_df_wide = dm.multi_day_var_from_pretrained_model(
    model_xgb_wide, forecast_feature_wide, calendar,
    MULTI_DAY_VAR_WINDOW, VAR_CONFIDENCE, run_key="xgb_wide",
)
var_df_wide.head()

In [ ]:
plot_var_vs_loss(var_df_wide, model_xgb_wide.residual_bank_in_sample, title="Wide: VaR Forecast vs Realized Loss")

## 7. Backtest

`calendar.backtest(key, var_confidence, test_confidence=0.95)`: `key` must be a `run_key` (a key of `calendar.var_prediction_runs`).

| Parameter | Meaning |
|---|---|
| `var_confidence` | Must be **the same** as the one used to produce the forecasts (the expected violation rate is `1 - var_confidence`) |
| `test_confidence` | Confidence level of the statistical tests (Kupiec, Christoffersen, etc.), default `0.95` |

In [ ]:
# >>> TUNABLE
TEST_CONFIDENCE = 0.95
BACKTEST_KEY    = "xgb_long"      # recorded keys: list(calendar.var_prediction_runs)
# <<<

print(list(calendar.var_prediction_runs))
calendar.backtest(BACKTEST_KEY, VAR_CONFIDENCE, TEST_CONFIDENCE)

## Quick summary: where each parameter is set and what it affects

| Parameter | Set in | Scope | Validity |
|---|---|---|---|
| `train_*`, `forecast_*` | Calendar | Everything (data volume, number of residuals) | Real bars, `train_end < forecast_start` |
| `drop_market_closed` | Calendar | Warning vs error on holiday gaps | – |
| `windows`, `ewma_span`, `sq_return_lags` | Feature builder | Feature set + warmup loss | Window ≥ 2 (for std / skew) |
| `target_scale` | Feature builder | Target scale (`y * scale`) | Positive |
| `model_kwargs` | `*_train` | Model capacity | Model-dependent |
| `rolling_mu_hat_window_size` | `*_train` → `ResidualBank` | `mu_hat` window (`rolling` only) | Positive int (required argument) |
| `mu_mode`, `default_mu_hat` | `*_train` → `ResidualBank` | Residual standardization and the `-mu_hat` term in VaR | `"rolling"` / `"constant"` |
| `ewma_lam`, `ewma_min_periods` | `long_format_train_seed` (not forwarded by `tree_train`) | Portfolio covariance on the long path | `0<lam<1`, `train_data >= min_periods` bars |
| `multi_day_var_calculation_window_size` | Forecast | VaR quantile window | Cannot exceed the bank's history |
| `var_confidence` | Forecast + Backtest | Quantile level `1 - conf` | `0 < conf < 1` |
| `run_key` | Forecast | Entry in `calendar.var_prediction_runs` | Overwrites if reused |
| `test_confidence` | Backtest | Confidence of the statistical tests | `0 < tc < 1` |